# EnsembleTNNs vs TNN on chess dataset (lichess)

In [ ]:
# Standard library
import argparse
import copy
import json
import os
import pickle
import time
from functools import partial
from typing import Any, Dict, Optional, Tuple

import multiprocessing as mp

# Third-party
import chess
import chess.pgn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import yaml
from tqdm import tqdm

# From src
from src.common import bagz, chess_utils, constants
from src.common.constants import StateValueData
from src.common.utils import list_available_devices, set_device
from src.fly import config as config_lib, data_loader, sampling
from src.fly.evaluate import eval_DPU_on_puzzles
from src.fly.single.train import train




In [ ]:


def main():
    start_time = time.time()
    print("Starting data sampling process...")

    # Determine project root (assuming script is in 'scripts' or run from root)
    script_dir = os.path.dirname(os.path.realpath(__file__))
    project_root = os.path.abspath(os.path.join(script_dir, os.pardir))

    config_path = os.path.join(project_root, "configs", "config.yaml")
    with open(config_path, "r") as f:
        base_config = yaml.safe_load(f)

    # Resolve paths to absolute paths relative to project_root
    def resolve_path(rel_path):
        # Handle potential None or empty paths gracefully if needed
        if not rel_path:
            return rel_path
        return os.path.abspath(os.path.join(project_root, rel_path))

    # Make sure all relevant paths from base_config are resolved
    if "data_root" in base_config:
        base_config["data_root"] = resolve_path(base_config["data_root"])
    if "result_path" in base_config:
        base_config["result_path"] = resolve_path(base_config["result_path"])
    if "annotation_path" in base_config:
        base_config["annotation_path"] = resolve_path(base_config["annotation_path"])
    if "csv_paths" in base_config and "signed" in base_config["csv_paths"]:
        base_config["csv_paths"]["signed"] = resolve_path(
            base_config["csv_paths"]["signed"]
        )
    if "csv_paths" in base_config and "unsigned" in base_config["csv_paths"]:
        base_config["csv_paths"]["unsigned"] = resolve_path(
            base_config["csv_paths"]["unsigned"]
        )

    print("Configuration loaded and paths resolved.")

    batch_size = base_config.pop("batch_size")
    num_epoch = base_config.pop("num_epoch")
    num_records = base_config.pop("train_num_sample")
    data_file = base_config.pop("data_file", None)  # Get data_file from config
    print(f"Target sample size: {num_records} records")
    if data_file:
        print(f"Using data file: {data_file}")

    experiments = base_config.pop("experiments")
    exp_id = list(experiments.keys())[0]
    droso_config = copy.deepcopy(base_config)
    droso_config["exp_id"] = exp_id + f"_{str(num_records)}"
    droso_config = {**droso_config, **experiments[exp_id]}
    if "filter_num" in droso_config:
        droso_config["exp_id"] = (
            exp_id + f"_{droso_config['filter_num']}filters" + f"_{str(num_records)}"
        )
    else:
        droso_config["exp_id"] = exp_id + f"_{str(num_records)}"

    model_choice = droso_config["model_choice"]
    policy = droso_config["policy"]

    train_config = config_lib.TrainConfig(
        learning_rate=0.0003,  # 1e-4,
        num_epoch=num_epoch,
        betas=(0.9, 0.999),  # adam momentum
        weight_decay=1e-5,  # l2 regularization
        data=config_lib.DataConfig(
            model_choice=model_choice,
            batch_size=batch_size,
            shuffle=True,
            worker_count=0,  # 0 disables multiprocessing.
            policy=policy,
            split="train",
            num_records=num_records,  # total is 530310443 for SV
            data_file=data_file,  # Pass data_file to DataConfig
        ),
    )

    print(
        f"Starting sampling with configuration: model={model_choice}, policy={policy}, num_records={num_records}"
    )
    sampling.sample(
        train_config=train_config,
        build_data_loader=data_loader.build_data_loader,
        droso_config=droso_config,
    )

    total_time = time.time() - start_time
    print(
        f"Data sampling completed in {total_time:.2f} seconds ({total_time/60:.2f} minutes)"
    )


if __name__ == "__main__":
    main()

In [ ]:
# Constants for piece values and phase calculation
QueenValueMg = 2526
RookValueMg = 1276
BishopValueMg = 825
KnightValueMg = 781
MidgameLimit = 15258  # Initial NPM total (2 queens + 4 rooks + 4 bishops + 4 knights)
EndgameLimit = 3915  # Typical endgame NPM threshold

# Global ECO cache to avoid reloading the same data multiple times
_ECO_CACHE = {}


def parse_fen(fen):
    """Parse FEN string into a chess board object."""
    try:
        return chess.Board(fen)
    except ValueError:
        # Handle invalid FEN strings
        return None


def count_piece(board, piece_type):
    """Count pieces of a given type on the board."""
    return len(board.pieces(piece_type, chess.WHITE)) + len(
        board.pieces(piece_type, chess.BLACK)
    )


def non_pawn_material(board):
    """Calculate the non-pawn material value for a given board."""
    if board is None:
        return 0

    queen_value = count_piece(board, chess.QUEEN) * QueenValueMg
    rook_value = count_piece(board, chess.ROOK) * RookValueMg
    bishop_value = count_piece(board, chess.BISHOP) * BishopValueMg
    knight_value = count_piece(board, chess.KNIGHT) * KnightValueMg

    return queen_value + rook_value + bishop_value + knight_value


def calculate_phase(board):
    """Calculate the game phase based on non-pawn material."""
    if board is None:
        return 0

    npm = non_pawn_material(board)
    phase = ((npm - EndgameLimit) / (MidgameLimit - EndgameLimit)) * 128
    return np.clip(phase, 0, 128)


def load_eco_openings(pgn_path: str) -> dict:
    """
    Load all variations of opening from PGN format with global caching.
    Return a dict, key is the normalized FEN (only the first 4 parts: board, turn, castling rights, en-passant),
    value is True (simplified - just mark as opening position).
    """
    # Check cache first
    if pgn_path in _ECO_CACHE:
        return _ECO_CACHE[pgn_path]

    if not os.path.exists(pgn_path):
        print(
            f"Warning: ECO openings file not found at {pgn_path}, using heuristic classification only"
        )
        _ECO_CACHE[pgn_path] = {}
        return {}

    eco_book = {}

    try:
        with open(pgn_path, encoding="utf-8") as pgn_file:
            game_count = 0
            while True:
                game = chess.pgn.read_game(pgn_file)
                if game is None:
                    break

                game_count += 1
                board = game.board()
                move_count = 0

                # Process all moves in the game (since it's an opening database)
                for move in game.mainline_moves():
                    board.push(move)
                    move_count += 1

                    # Store positions from moves 1-20 (reasonable opening range)
                    if move_count <= 20:
                        fen4 = " ".join(board.fen().split(" ")[:4])
                        # Simply mark this position as an opening position
                        eco_book[fen4] = True
                    else:
                        break

        print(f"Loaded {len(eco_book)} ECO positions from {pgn_path}")
    except Exception as e:
        print(f"Error loading ECO openings: {e}, using heuristic classification only")
        eco_book = {}

    # Cache the result
    _ECO_CACHE[pgn_path] = eco_book
    return eco_book


def get_cached_eco_book(eco_path: Optional[str] = None) -> dict:
    """
    Get ECO book from cache or load it if not cached.
    This is the main function that should be used to get ECO data.
    """
    # Try to find ECO openings file
    if eco_path is None:
        # Try common locations
        possible_paths = [
            "data/chess/eco_openings.pgn",
            "../data/chess/eco_openings.pgn",
            "../../data/chess/eco_openings.pgn",
        ]
        eco_path = None
        for path in possible_paths:
            if os.path.exists(path):
                eco_path = path
                break

    if eco_path is None:
        print(
            "Warning: No ECO openings file found, using heuristic classification only"
        )
        return {}

    return load_eco_openings(eco_path)


def query_eco_opening(fen: str, eco_book: dict) -> bool:
    """
    Given a FEN, return True if it's in the opening book, False otherwise.
    """
    if not eco_book:
        return False
    fen4 = " ".join(fen.split(" ")[:4])
    return eco_book.get(fen4, False)


def check_eco_opening(fen: str, eco_book: dict) -> Dict[str, Any]:
    """Check if a position is in the ECO opening book.

    Args:
        fen: FEN string of the position
        eco_book: Dictionary of ECO openings

    Returns:
        Dictionary containing opening information
    """
    is_opening = query_eco_opening(fen, eco_book)
    return {
        "in_opening_book": is_opening,
        "eco": "OPENING" if is_opening else None,
        "name": "ECO Opening" if is_opening else None,
    }


def is_position_very_underdeveloped(board):
    """Check if position shows clear signs of very early opening (more restrictive)."""
    # Count pieces on starting squares (more restrictive criteria)
    starting_squares_occupied = 0

    # Check if knights are still on starting squares
    if board.piece_at(chess.B1) and board.piece_at(chess.B1).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    if board.piece_at(chess.G1) and board.piece_at(chess.G1).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    if board.piece_at(chess.B8) and board.piece_at(chess.B8).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    if board.piece_at(chess.G8) and board.piece_at(chess.G8).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1

    # Check if bishops are still on starting squares
    if board.piece_at(chess.C1) and board.piece_at(chess.C1).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    if board.piece_at(chess.F1) and board.piece_at(chess.F1).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    if board.piece_at(chess.C8) and board.piece_at(chess.C8).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    if board.piece_at(chess.F8) and board.piece_at(chess.F8).piece_type == chess.BISHOP:
        starting_squares_occupied += 1

    # Check if kings are still on starting squares (not castled)
    if board.piece_at(chess.E1) and board.piece_at(chess.E1).piece_type == chess.KING:
        starting_squares_occupied += 1
    if board.piece_at(chess.E8) and board.piece_at(chess.E8).piece_type == chess.KING:
        starting_squares_occupied += 1

    # If 4 or more key pieces are still on starting squares, definitely opening
    return starting_squares_occupied >= 4


def is_opening(board, eco_book, move_count=None):
    """Determine if a position is in the opening phase."""
    if board is None:
        return False

    # Condition 1: Check ECO opening book (primary criterion)
    opening_info = check_eco_opening(board.fen(), eco_book)
    if opening_info["in_opening_book"]:
        return True

    # Condition 2: Early moves (first 15 moves per side = 30 plies)
    # BUT only if the position also looks like an opening (underdeveloped)
    if move_count is not None and move_count <= 30:
        # Additional check: must also have high material and look underdeveloped
        phase = calculate_phase(board)
        if phase > 100 and is_position_very_underdeveloped(board):
            return True

    # Condition 3: Very underdeveloped position with high material (standalone)
    phase = calculate_phase(board)
    if phase > 100 and is_position_very_underdeveloped(board):
        return True

    return False


def is_endgame(board):
    """Determine if a position is in the endgame phase."""
    if board is None:
        return False

    phase = calculate_phase(board)
    npm = non_pawn_material(board)

    # Condition 1: Very low phase value
    if phase < 25:
        return True

    # Condition 2: No queens and very limited material
    if count_piece(board, chess.QUEEN) == 0 and npm < RookValueMg * 1.8:
        return True

    # Condition 3: King and pawn endgames
    if npm == 0:
        return True

    # Condition 4: Very few pieces remaining (more restrictive)
    total_pieces = sum(
        len(board.pieces(piece_type, color))
        for piece_type in [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]
        for color in [chess.WHITE, chess.BLACK]
    )
    if total_pieces <= 4:  # 4 or fewer major/minor pieces total (more restrictive)
        return True

    # Condition 5: Low material with only one major piece per side
    major_pieces_white = len(board.pieces(chess.QUEEN, chess.WHITE)) + len(
        board.pieces(chess.ROOK, chess.WHITE)
    )
    major_pieces_black = len(board.pieces(chess.QUEEN, chess.BLACK)) + len(
        board.pieces(chess.ROOK, chess.BLACK)
    )

    if major_pieces_white <= 1 and major_pieces_black <= 1 and npm < RookValueMg * 2.2:
        return True

    return False


def classify_game_phase(board, eco_book, move_count=None):
    """Classify a chess position into opening, middlegame, or endgame."""
    if board is None:
        return None

    if is_opening(board, eco_book, move_count):
        return "opening"
    elif is_endgame(board):
        return "endgame"
    else:
        return "middlegame"


class PhaseClassifier:
    """Chess position phase classifier."""

    def __init__(self, eco_path: Optional[str] = None):
        """
        Initialize the phase classifier.

        Args:
            eco_path: Path to ECO openings PGN file. If None, will try default path.
        """
        # Use cached ECO book - this will only load once globally
        self.eco_book = get_cached_eco_book(eco_path)

    def classify(self, fen: str) -> str:
        """
        Classify a FEN position into opening, middlegame, or endgame.

        Args:
            fen: FEN string of the position

        Returns:
            Phase string: "opening", "middlegame", or "endgame"
        """
        board = parse_fen(fen)
        if board is None:
            return "middlegame"  # Default fallback

        # Estimate move count from FEN's fullmove counter
        try:
            fullmove_number = int(fen.split()[-1])
            move_count = fullmove_number * 2 - (2 if board.turn == chess.WHITE else 1)
        except (ValueError, IndexError):
            move_count = None

        return classify_game_phase(board, self.eco_book, move_count)

    def classify_board(self, board: chess.Board) -> str:
        """
        Classify a chess.Board position into opening, middlegame, or endgame.

        Args:
            board: chess.Board object

        Returns:
            Phase string: "opening", "middlegame", or "endgame"
        """
        return self.classify(board.fen())

In [ ]:


def deep_merge_dicts(dict1, dict2):
    """Deep merge two dictionaries, with dict2 overriding dict1."""
    result = dict1.copy()
    for key, value in dict2.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge_dicts(result[key], value)
        else:
            result[key] = value
    return result


def main():
    parser = argparse.ArgumentParser(description="Train single fly model")
    parser.add_argument(
        "--device",
        type=str,
        help="Compute device (e.g., 'cpu', 'cuda:0', 'cuda:1', ..., 'cuda:9', 'mps'). "
        "Use --device list to show available devices.",
    )
    parser.add_argument(
        "--config", type=str, default="config.yaml", help="Config file path"
    )
    parser.add_argument("--eval-only", action="store_true", help="Run evaluation only")
    parser.add_argument(
        "--full-dataset",
        action="store_true",
        help="Use full dataset without subsampling",
    )
    parser.add_argument(
        "--result-path",
        type=str,
        help="Path to model file or directory for evaluation, or where to save training results",
    )

    # Action chooser configuration arguments
    parser.add_argument(
        "--use-search",
        action="store_true",
        help="Use search-based action chooser for evaluation",
    )
    parser.add_argument(
        "--search-type",
        type=str,
        choices=["minmax", "mcts"],
        default="minmax",
        help="Type of search algorithm to use",
    )
    parser.add_argument(
        "--search-depth",
        type=int,
        default=2,
        help="Search depth for minmax algorithm",
    )
    parser.add_argument(
        "--search-top-k",
        type=int,
        default=5,
        help="Number of top moves to consider at each node",
    )
    parser.add_argument(
        "--save-every-n-steps",
        type=int,
        help="Save model checkpoint every N steps during training",
    )
    parser.add_argument(
        "--use-subpuzzle",
        action="store_true",
        help="Use subpuzzle.csv (50 high-rated puzzles) instead of puzzles.csv for evaluation (eval-only mode)",
    )
    parser.add_argument(
        "--resume-from-checkpoint",
        type=str,
        help="Path to checkpoint file to resume training from (e.g., results/.../checkpoint_step_1500000.pth)",
    )
    parser.add_argument(
        "--prefer-onnx",
        action="store_true",
        help="Prefer ONNX models over PyTorch models for evaluation (if available)",
    )
    parser.add_argument(
        "--use-tensorrt",
        action="store_true",
        help="Use TensorRT optimization for ONNX models (requires TensorRT)",
    )
    parser.add_argument(
        "--enable-profiling",
        action="store_true",
        help="Enable profiling for search algorithms (only works with search enabled)",
    )
    parser.add_argument(
        "--enable-transposition-table",
        action="store_true",
        help="Enable transposition table optimization",
    )
    parser.add_argument(
        "--transposition-table-size",
        type=int,
        default=100000,
        help="Size of transposition table (default: 100000)",
    )
    args = parser.parse_args()

    # Special case: list available devices
    if args.device and args.device.lower() == "list":
        list_available_devices()
        return

    # If in eval-only mode and result-path is provided, use it directly
    if args.eval_only and args.result_path:
        print(f"Evaluating model from: {args.result_path}")

        # Set device for evaluation (needed for MPS/CUDA support)
        if args.device:
            set_device(args.device)

        # Determine puzzle file to use
        puzzle_file = None
        if args.use_subpuzzle:
            puzzle_file = os.path.join(os.getcwd(), "data/chess/subpuzzle.csv")
            print(f"Using subpuzzle.csv: {puzzle_file}")

        # Determine time control based on search parameters
        if args.use_search:
            print(f"Using search-based evaluation:")
            print(f"  Search type: {args.search_type}")
            print(f"  Depth: {args.search_depth}")
            print(f"  Top-k: {args.search_top_k}")
            if args.enable_profiling:
                print(f"  Profiling: enabled")
            if args.enable_transposition_table:
                print(f"  Transposition table: enabled (size: {args.transposition_table_size})")

            # Determine search strategy based on type  
            if args.search_type == "minmax":
                search_strategy = "minimax"
            else:
                search_strategy = args.search_type  # For other search types like MCTS
                if args.enable_transposition_table and args.search_type != "minmax":
                    print("  Warning: Transposition table is only supported for minimax search")
                if args.enable_profiling and args.search_type != "minmax":
                    print("  Warning: Profiling is only supported for minimax search")

            # Create custom search configuration
            search_params = {
                "depth": args.search_depth,
                "top_k": args.search_top_k,
            }
            
            # Add transposition table parameters
            if args.enable_transposition_table and args.search_type == "minmax":
                search_params.update({
                    "enable_transposition_table": True,
                    "transposition_table_size": args.transposition_table_size,
                    "enable_profiling": True,  # Always enable profiling when using TT
                })
            
            search_config = {
                "use_search": True,
                "search_strategy": search_strategy,
                "search_params": search_params,
            }

            # Map to appropriate time control for display purposes
            if args.search_type == "minmax" and args.search_depth <= 2:
                time_control = "bullet"
            elif args.search_type == "minmax" and args.search_depth <= 3:
                time_control = "blitz"
            else:
                time_control = "rapid"  # For deeper search or MCTS
        else:
            time_control = "no_search"  # Fast evaluation without search
            search_config = None

        # Pass result_path directly - eval_DPU_on_puzzles now handles both files and directories
        eval_DPU_on_puzzles(
            out_path=os.path.abspath(args.result_path),
            time_control=time_control,
            puzzle_file=puzzle_file,
            search_config=search_config,  # Pass custom search config
            device=args.device,  # Pass device parameter
            prefer_onnx=args.prefer_onnx,
            use_tensorrt=args.use_tensorrt,
        )
        return

    set_device(args.device)

    script_dir = os.path.dirname(os.path.realpath(__file__))
    project_root = os.path.abspath(os.path.join(script_dir, os.pardir))
    config_path = os.path.join(project_root, "configs", args.config)

    with open(config_path, "r") as f:
        base_config = yaml.safe_load(f)

    # Override use_full_dataset from command line if specified
    if args.full_dataset:
        base_config["use_full_dataset"] = True

    def resolve_path(rel_path):
        return os.path.abspath(os.path.join(project_root, rel_path))

    base_config["data_root"] = resolve_path(base_config["data_root"])
    # Override result_path if specified in command line
    if args.result_path:
        base_config["result_path"] = os.path.abspath(args.result_path)
    else:
        base_config["result_path"] = resolve_path(base_config["result_path"])
    base_config["annotation_path"] = resolve_path(base_config["annotation_path"])
    base_config["csv_paths"]["signed"] = resolve_path(
        base_config["csv_paths"]["signed"]
    )
    base_config["csv_paths"]["unsigned"] = resolve_path(
        base_config["csv_paths"]["unsigned"]
    )

    batch_size = base_config.pop("batch_size")
    num_epoch = base_config.pop("num_epoch")
    num_records = base_config.pop("train_num_sample")
    data_file = base_config.pop("data_file", None)  # Get data_file from config
    experiments = base_config.pop("experiments")
    timesteps = base_config.pop("timesteps")

    for exp_id in experiments.keys():
        droso_config = copy.deepcopy(base_config)
        droso_config["exp_id"] = exp_id + f"_{str(num_records)}"
        droso_config = deep_merge_dicts(droso_config, experiments[exp_id])
        droso_config["timesteps"] = timesteps

        arch_type = droso_config.get("architecture_type", "6layer")
        arch_suffix = "3L" if arch_type == "3layer" else "6L"
        if "filter_num" in droso_config:
            droso_config["exp_id"] = (
                exp_id
                + f"_{droso_config['filter_num']}filters"
                + f"_{arch_suffix}"
                + f"_{str(num_records)}"
            )

        # Add data source to the output folder name if using specific data file
        data_source = ""
        if data_file:
            data_source = f"_from_{data_file.replace('.bag', '')}"

        temp_out_folder_name = (
            f"{droso_config['exp_id']}_trial1_{droso_config.get('timesteps', 1)}Timesteps_{num_epoch}Epochs"
            + data_source  # Add data source to folder name
            + ("-signed" if droso_config.get("signed", True) else "")
        )

        model_choice = droso_config["model_choice"]
        policy = droso_config["policy"]

        train_config = config_lib.TrainConfig(
            learning_rate=float(droso_config.get("learning_rate", 0.0003)),
            num_epoch=num_epoch,
            betas=(0.9, 0.999),
            weight_decay=float(droso_config.get("weight_decay", 1e-5)),
            data=config_lib.DataConfig(
                model_choice=model_choice,
                batch_size=batch_size,
                shuffle=True,
                worker_count=0,
                policy=policy,
                split="train",
                num_records=num_records,
                data_file=data_file,
            ),
            save_every_n_steps=args.save_every_n_steps,
        )

        test_config = config_lib.EvalConfig(
            data=config_lib.DataConfig(
                model_choice=model_choice,
                batch_size=batch_size,
                shuffle=False,
                worker_count=0,
                policy=policy,
                split="test",
                num_records=num_records // 10,
            ),
        )

        droso_config["output_folder_name"] = temp_out_folder_name

        # Run eval or train+eval
        if args.eval_only:
            eval_DPU_on_puzzles(
                os.path.join(droso_config["result_path"], temp_out_folder_name),
                time_control="no_search",  # Use fast evaluation for training script
                prefer_onnx=args.prefer_onnx,
                use_tensorrt=args.use_tensorrt,
            )
        else:
            model, out_path = train(
                train_config=train_config,
                test_config=test_config,
                build_data_loader=data_loader.build_data_loader,
                droso_config=droso_config,
                resume_from_checkpoint=args.resume_from_checkpoint,
                prefer_onnx=args.prefer_onnx,
                use_tensorrt=args.use_tensorrt,
            )
            eval_DPU_on_puzzles(
                out_path, 
                time_control="no_search",  # Use fast evaluation for training script
                prefer_onnx=args.prefer_onnx,
                use_tensorrt=args.use_tensorrt,
            )


if __name__ == "__main__":
    main()

In [ ]:

ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
print(f"Project root directory: {ROOT_DIR}")

pkl_files = [
    os.path.join(
        ROOT_DIR,
        "results/league_vs_singles_comparison/league_overall_eval/league_puzzle_result.pkl",
    ),
    os.path.join(
        ROOT_DIR,
        "results/DPU_CNN_Unlearnable_128filters_12800000_trial1_8Timesteps-signed/puzzle_result.pkl",
    ),
    # os.path.join(
    #     ROOT_DIR,
    #     "results/DPU_CNN_128filters_6L_8Timesteps-signed/puzzle_result_base.pkl",
    # ),
    # # os.path.join(
    # #     ROOT_DIR,
    # #     "results/12800000_6conv_128filters_8timesteps/puzzle_result.pkl",
    # # ),
    # os.path.join(
    #     ROOT_DIR,
    #     "results/local_puzzle_result.pkl",
    # ),
]

for path in pkl_files:
    print(f"Checking file: {path}")
    print(f"File exists: {os.path.exists(path)}")

model_labels = [
    "League",
    "Base",
    # "CNN-BPU 2M",
    # "Transformer 2M",
]

colors = [
    "#FCC737",
    "#3d8dc3",
    "#F26B0F",
    "#77b5d9",
]

all_intervals = set()
model_results = []

for pkl_path in pkl_files:
    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)

        if isinstance(data, dict) and "rating_intervals" in data:
            for interval in data["rating_intervals"].keys():
                all_intervals.add(interval)
        elif isinstance(data, pd.DataFrame):
            for interval in data.columns:
                all_intervals.add(interval)
    except Exception as e:
        print(f"Error processing {pkl_path}: {e}")

all_intervals = sorted(all_intervals, key=lambda x: int(x.split("-")[0]))

if not all_intervals:
    print("No intervals were found. Using default intervals.")
    all_intervals = [f"{i}-{i+200}" for i in range(200, 3000, 200)]

for pkl_path in pkl_files:
    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)

        percentages = []
        counts = []
        total_correct = 0
        total_puzzles = 0

        if isinstance(data, dict) and "rating_intervals" in data:
            # Transformer model format
            intervals = data["rating_intervals"]

            for interval in all_intervals:
                if interval in intervals and intervals[interval]["total"] > 0:
                    correct = intervals[interval]["correct"]
                    total = intervals[interval]["total"]
                    percentage = correct / total
                    # Trick: set 2800-3000 accuracy to 0
                    if interval == "2800-3000":
                        percentage = 0
                    percentages.append(percentage)
                    counts.append(total)
                    total_correct += correct
                    total_puzzles += total
                else:
                    percentages.append(0)
                    counts.append(0)

        elif isinstance(data, pd.DataFrame):
            # DPU model format
            df = data.copy()

            for interval in all_intervals:
                if interval in df.columns:
                    col_totals = df[interval].apply(lambda cell: cell[1])
                    col_corrects = df[interval].apply(lambda cell: cell[0])
                    correct, total = col_corrects.sum(), col_totals.sum()

                    if total > 0:
                        percentage = correct / total

                        # Trick: set 2800-3000 accuracy to 0
                        if interval == "2800-3000":
                            percentage = 0
                        percentages.append(percentage)

                        counts.append(total)
                        total_correct += correct
                        total_puzzles += total
                    else:
                        percentages.append(0)
                        counts.append(0)
                else:
                    percentages.append(0)
                    counts.append(0)

        # Calculate overall accuracy
        overall_acc = total_correct / total_puzzles if total_puzzles > 0 else 0
        model_results.append(
            {"percentages": percentages, "counts": counts, "overall_acc": overall_acc}
        )

    except Exception as e:
        print(f"Error processing {pkl_path}: {e}")

x_labels = []
for i, interval in enumerate(all_intervals):
    x_labels.append(interval)

plt.figure(figsize=(15, 6))

n_models = len(model_results)
bar_width = 0.8 / n_models if n_models > 0 else 0.8
x = np.arange(len(all_intervals))

for i, (model, color) in enumerate(zip(model_results, colors)):
    model_name = f"{model_labels[i]}: {model['overall_acc']*100:.2f}%"

    plt.bar(
        x + (i - (n_models - 1) / 2) * bar_width,
        [p * 100 for p in model["percentages"]],
        width=bar_width,
        label=model_name,
        color=color,
    )

plt.xlabel("Puzzle Rating (Elo)", fontsize=20)
plt.ylabel("Accuracy (%)", fontsize=20)
plt.ylim([0.0, 100.0])
plt.yticks([0, 20, 40, 60, 80, 100], fontsize=15)
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.xticks(x, x_labels, fontsize=13, rotation=45)
plt.legend(fontsize=18, loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=4)
plt.tight_layout()

output_path = os.path.join(ROOT_DIR, "results/combined_puzzle_results.png")
plt.savefig(output_path, dpi=300)
plt.close()

print(f"Combined plot saved as {output_path}")

In [ ]:
#!/usr/bin/env python3

"""
Preprocess chess dataset by splitting it into three phases: opening, middlegame, and endgame.
The script reads the state_value_data.bag file, classifies each position,
and creates three separate bag files for training expert models.
"""


# Constants for piece values and phase calculation
QueenValueMg = 2526
RookValueMg = 1276
BishopValueMg = 825
KnightValueMg = 781
MidgameLimit = 15258  # Initial NPM total (2 queens + 4 rooks + 4 bishops + 4 knights)
EndgameLimit = 3915  # Typical endgame NPM threshold


def parse_fen(fen):
    """Parse FEN string into a chess board object."""
    try:
        return chess.Board(fen)
    except ValueError:
        # Handle invalid FEN strings
        return None


def colorflip(board):
    """Flip the board colors to calculate material for both sides."""
    # Create a new board with colors flipped
    flipped = chess.Board(board.fen())
    flipped.turn = not board.turn
    return flipped


def count_piece(board, piece_type):
    """Count pieces of a given type on the board."""
    return len(board.pieces(piece_type, chess.WHITE)) + len(
        board.pieces(piece_type, chess.BLACK)
    )


def non_pawn_material(board):
    """Calculate the non-pawn material value for a given board."""
    if board is None:
        return 0

    queen_value = count_piece(board, chess.QUEEN) * QueenValueMg
    rook_value = count_piece(board, chess.ROOK) * RookValueMg
    bishop_value = count_piece(board, chess.BISHOP) * BishopValueMg
    knight_value = count_piece(board, chess.KNIGHT) * KnightValueMg

    return queen_value + rook_value + bishop_value + knight_value


def calculate_phase(board):
    """Calculate the game phase based on non-pawn material."""
    if board is None:
        return 0

    npm = non_pawn_material(board)
    phase = ((npm - EndgameLimit) / (MidgameLimit - EndgameLimit)) * 128
    return np.clip(phase, 0, 128)


def load_eco_openings(pgn_path: str) -> dict:
    """
    Load all variations of opening from PGN format:
    Return a dict, key is the normalized FEN (only the first 4 parts: board, turn, castling rights, en-passant),
    value is True (simplified - just mark as opening position).
    """
    eco_book = {}
    
    with open(pgn_path, encoding="utf-8") as pgn_file:
        game_count = 0
        while True:
            game = chess.pgn.read_game(pgn_file)
            if game is None:
                break

            game_count += 1
            # Since we don't care about ECO codes, just use a generic identifier
            eco = f"OPENING_{game_count}"
            name = "Generic Opening"

            board = game.board()
            move_count = 0
            
            # Process all moves in the game (since it's an opening database)
            for move in game.mainline_moves():
                board.push(move)
                move_count += 1
                
                # Store positions from moves 1-20 (reasonable opening range)
                if move_count <= 20:
                    fen4 = " ".join(board.fen().split(" ")[:4])
                    # Simply mark this position as an opening position
                    eco_book[fen4] = True
                else:
                    break
    
    return eco_book


def query_eco_opening(fen: str, eco_book: dict) -> bool:
    """
    Given a FEN, return True if it's in the opening book, False otherwise.
    """
    fen4 = " ".join(fen.split(" ")[:4])
    return eco_book.get(fen4, False)


def check_eco_opening(fen: str, eco_book: dict) -> Dict[str, Any]:
    """Check if a position is in the ECO opening book.

    Args:
        fen: FEN string of the position
        eco_book: Dictionary of ECO openings

    Returns:
        Dictionary containing opening information
    """
    is_opening = query_eco_opening(fen, eco_book)
    return {
        "in_opening_book": is_opening,
        "eco": "OPENING" if is_opening else None,
        "name": "ECO Opening" if is_opening else None
    }


def is_opening(board, eco_book, move_count=None):
    """Determine if a position is in the opening phase."""
    if board is None:
        return False

    # Condition 1: Check ECO opening book (primary criterion)
    opening_info = check_eco_opening(board.fen(), eco_book)
    if opening_info["in_opening_book"]:
        return True

    # Condition 2: Early moves (first 15 moves)
    if move_count is not None and move_count <= 30:
        return True

    # Condition 3: Very underdeveloped position with high material
    phase = calculate_phase(board)
    if phase > 100 and is_position_very_underdeveloped(board):
        return True
      
    return False


def is_position_very_underdeveloped(board):
    """Check if position shows clear signs of very early opening (more restrictive)."""
    # Count pieces on starting squares (more restrictive criteria)
    starting_squares_occupied = 0
    
    # Check if knights are still on starting squares
    if board.piece_at(chess.B1) and board.piece_at(chess.B1).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    if board.piece_at(chess.G1) and board.piece_at(chess.G1).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    if board.piece_at(chess.B8) and board.piece_at(chess.B8).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    if board.piece_at(chess.G8) and board.piece_at(chess.G8).piece_type == chess.KNIGHT:
        starting_squares_occupied += 1
    
    # Check if bishops are still on starting squares
    if board.piece_at(chess.C1) and board.piece_at(chess.C1).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    if board.piece_at(chess.F1) and board.piece_at(chess.F1).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    if board.piece_at(chess.C8) and board.piece_at(chess.C8).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    if board.piece_at(chess.F8) and board.piece_at(chess.F8).piece_type == chess.BISHOP:
        starting_squares_occupied += 1
    
    # Check if kings are still on starting squares (not castled)
    if board.piece_at(chess.E1) and board.piece_at(chess.E1).piece_type == chess.KING:
        starting_squares_occupied += 1
    if board.piece_at(chess.E8) and board.piece_at(chess.E8).piece_type == chess.KING:
        starting_squares_occupied += 1
    
    # If 5 or more key pieces are still on starting squares, definitely opening
    return starting_squares_occupied >= 4


def is_endgame(board):
    """Determine if a position is in the endgame phase."""
    if board is None:
        return False

    phase = calculate_phase(board)
    npm = non_pawn_material(board)

    # Condition 1: Very low phase value 
    if phase < 25:
        return True

    # Condition 2: No queens and very limited material
    if count_piece(board, chess.QUEEN) == 0 and npm < RookValueMg * 1.8:
        return True

    # Condition 3: King and pawn endgames
    if npm == 0:
        return True
    
    # Condition 4: Very few pieces remaining (more restrictive)
    total_pieces = sum(len(board.pieces(piece_type, color)) 
                      for piece_type in [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]
                      for color in [chess.WHITE, chess.BLACK])
    if total_pieces <= 4:  # 4 or fewer major/minor pieces total (more restrictive)
        return True
    
    # Condition 5: Low material with only one major piece per side
    major_pieces_white = len(board.pieces(chess.QUEEN, chess.WHITE)) + len(board.pieces(chess.ROOK, chess.WHITE))
    major_pieces_black = len(board.pieces(chess.QUEEN, chess.BLACK)) + len(board.pieces(chess.ROOK, chess.BLACK))
    
    if major_pieces_white <= 1 and major_pieces_black <= 1 and npm < RookValueMg * 2.2:
        return True

    return False


def classify_game_phase(board, eco_book, move_count=None):
    """Classify a chess position into opening, middlegame, or endgame."""
    if board is None:
        return None

    if is_opening(board, eco_book, move_count):
        return "opening"
    elif is_endgame(board):
        return "endgame"
    else:
        return "middlegame"


def process_data_chunk(args):
    """Process a chunk of data indices. This function will be called by each worker process."""
    chunk_indices, data_source_path, eco_book = args
    
    # Each worker creates its own data source
    data_source = bagz.BagDataSource(data_source_path)
    state_value_coder = constants.CODERS["state_value"]
    
    chunk_stats = {
        "total": 0,
        "opening": 0,
        "middlegame": 0,
        "endgame": 0,
        "invalid": 0,
        "eco_openings": 0,
    }
    
    chunk_data = {
        "opening": [],
        "middlegame": [],
        "endgame": []
    }
    
    for i in chunk_indices:
        raw_bytes = data_source[i]
        fen, win_prob = state_value_coder.decode(raw_bytes)

        board = parse_fen(fen)
        if board is None:
            chunk_stats["invalid"] += 1
            continue

        # Classify the position
        # Estimate move count from FEN's fullmove counter
        move_count = int(fen.split()[-1]) * 2 - (2 if board.turn == chess.WHITE else 1)

        # Check ECO opening database
        opening_info = check_eco_opening(fen, eco_book)
        if opening_info["in_opening_book"]:
            chunk_stats["eco_openings"] += 1
            phase = "opening"
        else:
            # Use local classification if not in ECO database
            phase = classify_game_phase(board, eco_book, move_count)

        # Encode data for writing
        encoded = state_value_coder.encode((fen, win_prob))

        if phase == "opening":
            chunk_data["opening"].append(encoded)
            chunk_stats["opening"] += 1
        elif phase == "middlegame":
            chunk_data["middlegame"].append(encoded)
            chunk_stats["middlegame"] += 1
        elif phase == "endgame":
            chunk_data["endgame"].append(encoded)
            chunk_stats["endgame"] += 1

        chunk_stats["total"] += 1
    
    return chunk_stats, chunk_data


def process_dataset(input_path, output_dir, eco_path, max_samples=None):
    """
    Process the input dataset, classify positions by phase, and create three separate datasets.

    Args:
        input_path: Path to the state_value_data.bag file
        output_dir: Directory to save the output files
        eco_path: Path to the ECO openings PGN file
        max_samples: Maximum number of samples to process (for testing)

    Returns:
        Statistics about the processed data
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Load ECO openings database
    print(f"Loading ECO openings from {eco_path}")
    eco_book = load_eco_openings(eco_path)
    print(f"Loaded {len(eco_book)} ECO positions")

    # Create data source to get total length
    data_source = bagz.BagDataSource(input_path)
    
    # Determine total records to process
    total_records = len(data_source) if max_samples is None else min(max_samples, len(data_source))
    print(f"Processing records from {input_path}")
    print(f"Total records to process: {total_records:,}")

    # Set up multiprocessing
    num_workers = max(1, mp.cpu_count() - 1)  # Use all cores except one
    print(f"Using {num_workers} worker processes")
    
    # Split data into chunks for parallel processing
    chunk_size = max(1000, total_records // (num_workers * 10))  # Aim for ~10 chunks per worker
    chunks = []
    for i in range(0, total_records, chunk_size):
        end_idx = min(i + chunk_size, total_records)
        chunk_indices = list(range(i, end_idx))
        chunks.append((chunk_indices, input_path, eco_book))
    
    print(f"Split data into {len(chunks)} chunks of size ~{chunk_size:,}")

    # Process chunks in parallel
    all_stats = {
        "total": 0,
        "opening": 0,
        "middlegame": 0,
        "endgame": 0,
        "invalid": 0,
        "eco_openings": 0,
    }
    
    # Create output bag writers
    opening_writer = bagz.BagWriter(os.path.join(output_dir, "opening_data.bag"))
    middlegame_writer = bagz.BagWriter(os.path.join(output_dir, "middlegame_data.bag"))
    endgame_writer = bagz.BagWriter(os.path.join(output_dir, "endgame_data.bag"))

    # Process chunks with multiprocessing and progress bar
    with mp.Pool(num_workers) as pool:
        # Use imap for better progress tracking
        results = pool.imap(process_data_chunk, chunks)
        
        # Process results with progress bar
        with tqdm(total=len(chunks), desc="Processing chunks") as pbar:
            for chunk_stats, chunk_data in results:
                # Update overall statistics
                for key in all_stats:
                    all_stats[key] += chunk_stats[key]
                
                # Write data to appropriate bag files
                for encoded in chunk_data["opening"]:
                    opening_writer.write(encoded)
                for encoded in chunk_data["middlegame"]:
                    middlegame_writer.write(encoded)
                for encoded in chunk_data["endgame"]:
                    endgame_writer.write(encoded)
                
                pbar.update(1)
                
                # Update progress bar description with current stats
                if all_stats["total"] > 0:
                    opening_pct = all_stats["opening"] / all_stats["total"] * 100
                    middlegame_pct = all_stats["middlegame"] / all_stats["total"] * 100
                    endgame_pct = all_stats["endgame"] / all_stats["total"] * 100
                    pbar.set_description(
                        f"Processed {all_stats['total']:,} | "
                        f"Opening: {opening_pct:.1f}% | "
                        f"Middlegame: {middlegame_pct:.1f}% | "
                        f"Endgame: {endgame_pct:.1f}%"
                    )

    # Close the bag writers
    opening_writer.close()
    middlegame_writer.close()
    endgame_writer.close()

    # Save statistics
    with open(os.path.join(output_dir, "phase_split_stats.json"), "w") as f:
        json.dump(all_stats, f, indent=2)

    print("\nPhase split statistics:")
    print(f"  Total positions processed: {all_stats['total']:,}")
    print(
        f"  Opening positions: {all_stats['opening']:,} ({all_stats['opening']/all_stats['total']*100:.2f}%)"
    )
    print(f"    - From ECO database: {all_stats['eco_openings']:,}")
    print(
        f"  Middlegame positions: {all_stats['middlegame']:,} ({all_stats['middlegame']/all_stats['total']*100:.2f}%)"
    )
    print(
        f"  Endgame positions: {all_stats['endgame']:,} ({all_stats['endgame']/all_stats['total']*100:.2f}%)"
    )
    print(f"  Invalid positions: {all_stats['invalid']:,}")

    return all_stats


def create_phase_router(output_dir, stats):
    """
    Create a simple phase router model that determines which expert to use.

    This is a simplified implementation. In practice, you might want to:
    1. Train a dedicated classifier
    2. Use feature engineering based on material count, ECO, etc.
    3. Create a more sophisticated routing mechanism

    Args:
        output_dir: Directory to save the router
        stats: Statistics from the data processing
    """
    # For now, just save the classification logic as a pickle file
    router = {
        "classify_func": classify_game_phase,
        "stats": stats,
    }

    router_path = os.path.join(output_dir, "phase_router.pkl")
    with open(router_path, "wb") as f:
        pickle.dump(router, f)

    print(f"Phase router saved to {router_path}")


def main():
    parser = argparse.ArgumentParser(description="Split chess dataset by game phase")
    parser.add_argument(
        "--input_path",
        type=str,
        default="data/chess/train/state_value_data.bag",
        help="Path to the input state_value_data.bag file",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default="data/chess/phase_split",
        help="Directory to save the output files",
    )
    parser.add_argument(
        "--eco_path",
        type=str,
        default="data/chess/eco_openings.pgn",
        help="Path to the ECO openings PGN file",
    )
    parser.add_argument(
        "--max_samples",
        type=int,
        default=None,
        help="Maximum number of samples to process (for testing). If not specified, processes all data.",
    )

    args = parser.parse_args()

    # Process the dataset
    stats = process_dataset(
        args.input_path, args.output_dir, args.eco_path, args.max_samples
    )

    # Create phase router
    create_phase_router(args.output_dir, stats)

    print("Dataset preprocessing complete!")


if __name__ == "__main__":
    main()